In [12]:
import sqlite3
import pandas as pd

DB_PATH = "db/users_.db"

In [10]:
df = pd.read_csv('Data/Daniooo/Processed/final_output.csv')

In [27]:
df[(df['Tags']=='Food') & (df['Amount']>70)]

,Date,Desc,Run_balance,Amount,Tags
19,2024-03-09,Dop. Visa 421352******9407 Płatność Kartą 73.6...,499.86,73.69,Food
22,2024-03-11,Dop. Visa 421352******9407 Płatność Kartą 79.6...,324.38,79.67,Food
37,2024-03-23,Dop. Visa 421352******9407 Płatność Kartą 93.8...,506.72,93.88,Food
75,2024-04-15,Dop. Visa 421352******9407 Płatność Kartą 84.7...,246.48,84.75,Food
90,2024-04-27,Dop. Visa 421352******9407 Płatność Kartą 86.7...,502.17,86.74,Food
131,2024-05-20,Dop. Visa 421352******9407 Płatność Kartą 121....,299.60,121.45,Food
156,2024-06-10,Dop. Visa 421352******9407 Płatność Kartą 101....,379.34,101.86,Food
183,2024-07-29,Dop. Visa 421352******9407 Płatność Kartą 70.4...,400.71,70.49,Food
199,2024-08-05,Dop. Visa 421352******9407 Płatność Kartą 76.7...,1234.05,76.70,Food
210,2024-08-12,Dop. Visa 421352******9407 Płatność Kartą 74.6...,849.54,74.62,Food



    login_ TEXT NOT NULL,
    Date DATE,
    Desc TEXT,
    Run_balance REAL,
    Amount REAL,
    Tags TEXT,


In [17]:
def post_trans_data(role_):
    conn = sqlite3.connect(DB_PATH, timeout=30)
    # getting to all
    df_csv = pd.read_csv(f'Data/{role_}/Processed/final_output.csv')
    # foreign key column for db
    df_csv['login_'] = role_
    df_sql = pd.read_sql_query("SELECT * FROM trans", conn)
    df_new = pd.concat([df_sql,df_csv]).drop_duplicates(keep=False)
    df_new.to_sql(name='trans', con=conn, if_exists='append', index=False)

    conn.close()
    return 

In [18]:
post_trans_data('Daniooo')

In [5]:
import os
import sqlite3

path = "db/users_.db"
print("Using file:", os.path.abspath(path))
print("File exists:", os.path.exists(path))

with sqlite3.connect(path) as conn:
    conn.execute("""
        CREATE VIEW IF NOT EXISTS users_temp1 AS
        SELECT login_, seed
        FROM users
    """)

    print(conn.execute(
        "SELECT name, type FROM sqlite_master"
    ).fetchall())

    print(conn.execute("SELECT * FROM users_temp1 LIMIT 5").fetchall())

Using file: /Users/daniel_mac/Desktop/Projects/Budget_app/transaction_analysis_v2/db/users_.db
File exists: True
[('price_history', 'table'), ('user_portfolio', 'table'), ('users', 'table'), ('users_temp1', 'view')]
[('', 83), ('Daniel', 55), ('Danie', 1), ('', 82), ('Dan', 40)]


In [31]:
conn = sqlite3.connect(DB_PATH, timeout=30)
df_sql = pd.read_sql_query("SELECT * FROM trans WHERE login_ = ?", conn, params=('Daniooo',))
conn.close()

In [33]:
df_sql

,login_,Date,Desc,Run_balance,Amount,Tags
0,Daniooo,2024-02-29,Zakup Blik Jmp S.a. Biedronka Jmp S.a. Biedron...,566.98,69.50,Food
1,Daniooo,2024-02-29,JedzonkoMarzec636.48,636.48,0.00,Inflows
2,Daniooo,2024-03-01,Zakup Blik Zabka Z7537 K.2 Ul. Krakowska 46 Re...,1219.99,16.99,Food
3,Daniooo,2024-03-01,ZakupBlikSorellaKazimierzaWielkiego2Ref:794734...,1141.99,78.00,Others
4,Daniooo,2024-03-01,JedzenieP670.00,1236.98,670.00,Inflows
...,...,...,...,...,...,...
395,Daniooo,2024-12-24,BLIK - transakcja internetowa,276.88,9.55,Commute
396,Daniooo,2024-12-27,BLIK - transakcja internetowa,NaN,NaN,Commute
397,Daniooo,2024-12-27,OPŁATA ZA RACHUNEK/PAKIET,NaN,NaN,Others
398,Daniooo,2024-12-30,TRANS.BEZGOT. KARTĄ DEBET.,18.70,147.55,Food


In [105]:
df_sql['Date'] = pd.to_datetime(df_sql['Date'])
df_sql["Day"] = df_sql["Date"].dt.day
df_sql['Month'] = df_sql['Date'].dt.month
df_sql['Year'] = df_sql['Date'].dt.year
df_sql["Year-Month"] = pd.to_datetime(df_sql['Year'].astype(str) + '-' + df_sql['Month'].astype(str), format='%Y-%m').dt.strftime('%Y-%m')
df_sql['MTD'] = pd.to_datetime(df_sql['Year'].astype(str) + '-' + (df_sql['Month']-1).astype(str), format='%Y-%m').dt.strftime('%Y-%m')
df_sql['YTD'] = pd.to_datetime((df_sql['Year']-1).astype(str) + '-' + df_sql['Month'].astype(str), format='%Y-%m').dt.strftime('%Y-%m')
df_sql['WeekNo'] = df_sql['Date'].dt.isocalendar().week

In [ ]:
index_= df_sql['Year-Month'].drop_duplicates().to_list().index('2024-03')-1

0

In [103]:
df_sql['Month']-1

0       1
1       1
2       2
3       2
4       2
       ..
395    11
396    11
397    11
398    11
399    11
Name: Month, Length: 400, dtype: int64

In [66]:
df_ = df_sql['Year-Month'].drop_duplicates().reset_index(drop=True)

In [78]:
df_.shift()

0        None
1     2024-02
2     2024-03
3     2024-04
4     2024-05
5     2024-06
6     2024-07
7     2024-08
8     2024-09
9     2024-10
10    2024-11
Name: Year-Month, dtype: object

In [85]:
df_sql[df_sql['Year-Month']==max(df_sql['Year-Month'])]

,login_,Date,Desc,Run_balance,Amount,Tags,Day,Month,Year,WeekNo,Year-Month
358,Daniooo,2024-12-02,TRANS.BEZGOT. KARTĄ DEBET.,4.00,5.18,Beauty,2,12,2024,49,2024-12
359,Daniooo,2024-12-02,e-Platność,NaN,NaN,Others,2,12,2024,49,2024-12
360,Daniooo,2024-12-02,PRZELEW OTRZYMANY,NaN,NaN,Others,2,12,2024,49,2024-12
361,Daniooo,2024-12-05,PRZELEW OTRZYMANY ELIXIR,1027.00,1023.00,Inflows,5,12,2024,49,2024-12
362,Daniooo,2024-12-06,TRANS.BEZGOT. KARTĄ DEBET.,1015.40,11.60,House appliencies,6,12,2024,49,2024-12
363,Daniooo,2024-12-06,BLIK - transakcja internetowa,996.30,19.10,Commute,6,12,2024,49,2024-12
364,Daniooo,2024-12-09,TRANS.BEZGOT. KARTĄ DEBET.,967.66,24.85,House appliencies,9,12,2024,50,2024-12
365,Daniooo,2024-12-09,TRANS.BEZGOT. KARTĄ DEBET.,992.51,3.79,Others,9,12,2024,50,2024-12
366,Daniooo,2024-12-09,BLIK - transakcja internetowa,948.56,19.10,Commute,9,12,2024,50,2024-12
367,Daniooo,2024-12-10,TRANS.BEZGOT. KARTĄ DEBET.,884.83,0.65,Food,10,12,2024,50,2024-12


In [87]:
max(df_sql['Year-Month']).pct_change

AttributeError: 'str' object has no attribute 'pct_change'

In [ ]:
df['Quantity-1'] = df['Quantity'].shift(14)
   